# Project: Quantization Performance Optimization

Apply quantization techniques to your domain search engine and measure the real-world impact on speed, memory, and accuracy. You’ll discover how different quantization methods affect your specific use case and learn to optimize the accuracy recovery pipeline.

## Mission

Transform your search engine from previous days into a production-ready system by implementing quantization optimization. You’ll test different quantization methods, measure performance impacts, and tune the oversampling + rescoring pipeline for optimal results.

## What to Build

A quantization-optimized search system that demonstrates:

- Performance comparison: Before and after quantization metrics
- Method evaluation: Testing scalar and binary quantization on your data
- Accuracy recovery: Implementing oversampling and rescoring pipeline
- Production deployment: Memory-optimized storage configuration

In [1]:
pip install --upgrade ipywidgets

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 1: Baseline Measurement

In [11]:
# Example: Recipe collection
my_domain_collection = [
    {
        "title": "Classic Beef Bourguignon",
        "description": """A rich, wine-braised beef stew from Burgundy, France.
        Tender chunks of beef are slowly simmered with pearl onions, mushrooms,
        and bacon in a deep red wine sauce. The long, slow cooking process
        develops complex flavors and creates a luxurious, velvety texture.
        Perfect for cold winter evenings when you want something hearty and
        comforting. Traditionally served with crusty bread or creamy mashed
        potatoes to soak up the incredible sauce.""",
        "cuisine": "French",
        "difficulty": "Intermediate",
        "time": "3 hours"
    },
    {
        "title": "Thai Green Curry with Chicken",
        "description": """An aromatic and vibrant curry from Thailand featuring tender
        chicken in a coconut milk base infused with fragrant green curry paste, Thai basil,
        and kaffir lime leaves. The sauce balances spicy, sweet, and savory notes with
        bamboo shoots, eggplant, and bell peppers adding texture. Quick to prepare yet
        bursting with complex flavors, this dish brings the authentic taste of Bangkok
        street food to your kitchen. Serve over jasmine rice to soak up every drop of
        the creamy, spice-laden sauce.""",
        "cuisine": "Thai",
        "difficulty": "Easy",
        "time": "30 minutes"
    },
    {
        "title": "Homemade Margherita Pizza",
        "description": """The quintessential Neapolitan pizza that celebrates simplicity
        and quality ingredients. A thin, chewy crust with charred bubbles holds a bright
        San Marzano tomato sauce, creamy fresh mozzarella, and fragrant basil leaves.
        The key is high heat and minimal toppings, allowing each element to shine.
        Making the dough from scratch requires patience as it slowly ferments, developing
        complex flavors and that signature airy texture. Perfect for weekend cooking
        when you want to impress with authentic Italian technique.""",
        "cuisine": "Italian",
        "difficulty": "Intermediate",
        "time": "2 hours (plus dough rise)"
    },
    {
        "title": "Miso-Glazed Salmon with Sesame Vegetables",
        "description": """A modern Japanese-inspired dish that's both elegant and nutritious.
        Fresh salmon fillets are marinated in a sweet-savory miso glaze with mirin and sake,
        then broiled until caramelized and slightly charred at the edges. The umami-rich
        glaze creates a beautiful lacquered finish. Paired with crisp stir-fried vegetables
        tossed in sesame oil and garnished with toasted sesame seeds. This restaurant-quality
        meal comes together in under 25 minutes, making it perfect for busy weeknights when
        you don't want to sacrifice flavor or presentation.""",
        "cuisine": "Japanese",
        "difficulty": "Easy",
        "time": "25 minutes"
    },
    {
        "title": "Moroccan Lamb Tagine with Apricots",
        "description": """A fragrant North African stew that marries tender lamb with sweet
        dried apricots, aromatic spices, and preserved lemons. Slow-cooked in a traditional
        cone-shaped tagine or heavy pot, the meat becomes fall-apart tender while absorbing
        the warm spices of cinnamon, cumin, and ginger. Chickpeas add heartiness, while
        honey and apricots provide a delicate sweetness that balances the savory depth.
        The long, gentle cooking process allows the complex spice blend to fully develop.
        Serve over fluffy couscous with fresh cilantro and toasted almonds for a truly
        exotic dining experience.""",
        "cuisine": "Moroccan",
        "difficulty": "Intermediate",
        "time": "2.5 hours"
    },
    {
        "title": "Classic Chicken Caesar Salad",
        "description": """A timeless American restaurant staple that's surprisingly easy to
        master at home. Crisp romaine lettuce is tossed with a bold, creamy dressing made
        from anchovies, garlic, lemon, egg yolk, and Parmesan cheese. Topped with juicy
        grilled chicken breast, crunchy house-made croutons, and extra shaved Parmesan.
        The key is the dressing—punchy, garlicky, and perfectly emulsified. While often
        considered simple, a well-executed Caesar showcases the power of balancing strong,
        complementary flavors. Great for a light lunch or dinner that feels both indulgent
        and refreshing.""",
        "cuisine": "American",
        "difficulty": "Easy",
        "time": "20 minutes"
    },
    {
        "title": "Indian Butter Chicken (Murgh Makhani)",
        "description": """A beloved North Indian classic featuring tender chicken in a
        luxuriously creamy tomato-based sauce. The chicken is first marinated in yogurt
        and spices, then grilled or pan-fried for a smoky char before being simmered in
        a velvety sauce enriched with butter, cream, and aromatic spices like garam masala,
        fenugreek, and cardamom. The result is a perfect balance of tangy, sweet, and
        mildly spiced flavors with a silky texture. This restaurant favorite is easier to
        make at home than you'd think, and the aroma while cooking will transport you to
        the bustling streets of Delhi. Serve with naan bread and basmati rice.""",
        "cuisine": "Indian",
        "difficulty": "Intermediate",
        "time": "1 hour"
    },
    {
        "title": "Spanish Paella Valenciana",
        "description": """The iconic rice dish from Valencia that's meant for sharing and
        celebrating. Saffron-infused short-grain rice is cooked with chicken, rabbit, and
        green beans in a wide, shallow pan until it develops the prized 'socarrat'—a
        crispy, caramelized bottom layer. Fresh rosemary and smoked paprika add depth,
        while the saffron lends its distinctive golden color and earthy aroma. Traditionally
        cooked over an open fire, this one-pan feast brings people together. Making paella
        is as much about the ritual and patience as it is about the ingredients. The key
        is resisting the urge to stir, allowing those delicious crispy bits to form.""",
        "cuisine": "Spanish",
        "difficulty": "Advanced",
        "time": "1.5 hours"
    },
    {
        "title": "Vietnamese Pho Bo (Beef Noodle Soup)",
        "description": """Vietnam's national dish—a deeply aromatic beef broth that takes
        hours to perfect. Beef bones are simmered with charred onions, ginger, star anise,
        cinnamon, and coriander seeds until the broth becomes rich, clear, and intensely
        flavorful. Served over silky rice noodles with thinly sliced rare beef that cooks
        in the steaming broth, then finished with fresh herbs, lime, jalapeños, and bean
        sprouts. Each bowl is customizable at the table, making it interactive and personal.
        While time-intensive, the reward is a soul-warming bowl of pure comfort that rivals
        any pho shop in Hanoi.""",
        "cuisine": "Vietnamese",
        "difficulty": "Advanced",
        "time": "4 hours"
    },
    {
        "title": "Greek Moussaka",
        "description": """A hearty, layered casserole that's Greece's answer to lasagna.
        Tender slices of eggplant and potato are layered with a rich, spiced ground lamb
        and tomato sauce flavored with cinnamon, oregano, and a hint of red wine. The
        crown jewel is the creamy béchamel sauce on top, which bakes to a golden brown.
        Each forkful delivers multiple textures and layers of Mediterranean flavor. While
        it requires some prep work and patience, moussaka is perfect for feeding a crowd
        or meal prepping for the week. The flavors actually improve after a day, making
        leftovers even more delicious.""",
        "cuisine": "Greek",
        "difficulty": "Intermediate",
        "time": "2 hours"
    },
    {
        "title": "Korean Bibimbap with Gochujang",
        "description": """A colorful Korean rice bowl that's as beautiful as it is delicious.
        Warm rice is topped with an array of seasoned vegetables—sautéed spinach, carrots,
        bean sprouts, mushrooms—along with marinated beef, a fried egg, and a generous
        dollop of spicy-sweet gochujang sauce. The name means 'mixed rice,' and the ritual
        of stirring everything together before eating is essential. Each ingredient is
        prepared separately, showcasing different cooking techniques and seasonings. The
        result is a harmonious bowl where every bite offers different flavors and textures.
        Customizable and nutritious, it's perfect for using up vegetables and experimenting
        with Korean flavors.""",
        "cuisine": "Korean",
        "difficulty": "Intermediate",
        "time": "1 hour"
    },
    {
        "title": "Mexican Carnitas Tacos",
        "description": """Authentic slow-cooked pork that's tender on the inside with
        crispy, caramelized edges. A pork shoulder is braised low and slow in its own
        fat with orange juice, garlic, and Mexican spices until it's so tender it falls
        apart with a fork. Then it's crisped under the broiler for textural contrast.
        Tucked into warm corn tortillas and topped with fresh cilantro, diced onions,
        lime, and your favorite salsa. These tacos are a weekend project worth every
        minute—the kind of food that brings everyone to the table. Serve with refried
        beans and Mexican rice for an authentic taqueria experience at home.""",
        "cuisine": "Mexican",
        "difficulty": "Easy",
        "time": "3.5 hours (mostly hands-off)"
    }
]

In [23]:
# Delete existing quantized collections if they exist
for method_name in quantization_configs.keys():
    collection_name = f"quantized_{method_name}"
    try:
        client.delete_collection(collection_name=collection_name)
        print(f"✓ Deleted old {collection_name}")
    except:
        print(f"  {collection_name} doesn't exist, skipping delete")


✓ Deleted old quantized_scalar
✓ Deleted old quantized_binary
✓ Deleted old quantized_product


In [24]:
# Step 1: Delete old collections
for method_name in quantization_configs.keys():
    collection_name = f"quantized_{method_name}"
    try:
        client.delete_collection(collection_name=collection_name)
        print(f"✓ Deleted old {collection_name}")
    except:
        pass

✓ Deleted old quantized_scalar
✓ Deleted old quantized_binary
✓ Deleted old quantized_product


In [25]:
# Step 2: Create new collections with correct dimensions
for method_name, quantization_config in quantization_configs.items():
    collection_name = f"quantized_{method_name}"
    
    client.create_collection(
        collection_name=collection_name,
        vectors_config=models.VectorParams(
            size=384,
            distance=models.Distance.COSINE
        ),
        quantization_config=quantization_config
    )
    
    # Upload data
    points = []
    for i, recipe in enumerate(my_domain_collection):
        vector = encoder.encode(recipe["description"]).tolist()
        points.append(models.PointStruct(id=i, vector=vector, payload=recipe))
    
    client.upload_points(collection_name=collection_name, points=points)
    
    print(f"✓ Created and populated {collection_name}")


✓ Created and populated quantized_scalar
✓ Created and populated quantized_binary
✓ Created and populated quantized_product


In [16]:
my_test_queries = [
    # Semantic/conceptual queries (benefit from dense embeddings)
    "quick weeknight dinner",
    "comforting winter meal",
    "light and fresh lunch option",
    "impressive dish for guests",
    "soul-warming comfort food",
    
    # Keyword-specific queries (benefit from sparse/BM25)
    "French beef wine sauce",
    "Thai coconut curry chicken",
    "miso glazed salmon",
    "lamb apricots tagine",
    
    # Hybrid queries (benefit from both)
    "easy Asian recipe under 30 minutes",
    "slow-cooked Mexican pork tacos",
    "creamy Italian tomato cheese",
    "spicy rice bowl with vegetables",
    
    # Cuisine-specific
    "authentic Japanese seafood",
    "traditional Spanish rice dish",
    "North Indian creamy curry",
    
    # Time/difficulty constraints
    "quick recipe less than 30 minutes",
    "advanced technique multi-step cooking",
    "intermediate difficulty Mediterranean",
    
    # Ingredient-focused
    "lamb with dried fruit",
    "salmon with Asian flavors",
    "chicken with coconut and herbs"
]


In [26]:
import time
import numpy as np
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient, models
import os

client = QdrantClient(url=os.getenv("QDRANT_URL"), api_key=os.getenv("QDRANT_API_KEY"))

# For Colab:
# from google.colab import userdata
# client = QdrantClient(url=userdata.get("QDRANT_URL"), api_key=userdata.get("QDRANT_API_KEY"))

# Load the same encoder you used for indexing
encoder = SentenceTransformer("all-MiniLM-L6-v2")

def measure_search_performance(collection_name, test_queries, label="Baseline"):
    """Measure search performance across multiple queries"""
    latencies = []

    # Don't forget to warm up caches!

    #response = client.query_points(
    #        collection_name=collection_name,
    #        query=query,
    #        limit=10
    #    )
    
    for query_text in test_queries:
        # Encode the text query into a vector
        query_vector = encoder.encode(query_text).tolist()
        
        start_time = time.time()
        
        response = client.query_points(
            collection_name=collection_name,
            query=query_vector,
            limit=10
        )
        
        latency = (time.time() - start_time) * 1000  # ← Inside the loop
        latencies.append(latency)                     # ← Inside the loop
    
    # These lines are inside the function but AFTER the loop
    avg_latency = np.mean(latencies)
    p95_latency = np.percentile(latencies, 95)
    
    print(f"{label}:")
    print(f"  Average latency: {avg_latency:.2f}ms")
    print(f"  P95 latency: {p95_latency:.2f}ms")
    
    return {"avg": avg_latency, "p95": p95_latency}

# Measure baseline performance
baseline_metrics = measure_search_performance(
    "my_domain_collection", 
    my_test_queries, 
    "Baseline (No Quantization)"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Baseline (No Quantization):
  Average latency: 929.36ms
  P95 latency: 1975.89ms


## Step 2: Test Quantization Methods

In [31]:
quantization_configs = {
    "scalar": models.ScalarQuantization(
        scalar=models.ScalarQuantizationConfig(
            type=models.ScalarType.INT8,
            quantile=0.99,
            always_ram=True
        )
    ),
    "binary": models.BinaryQuantization(
        binary=models.BinaryQuantizationConfig(
            always_ram=True
        )
    ),
    "product": models.ProductQuantization(
        product=models.ProductQuantizationConfig(
            compression=models.CompressionRatio.X16,
            always_ram=True
        )
    )
}

# Create each quantized collection
for method_name, quantization_config in quantization_configs.items():
    collection_name = f"quantized_{method_name}"
    
    # DELETE FIRST - Add this section
    try:
        client.delete_collection(collection_name=collection_name)
        print(f"✓ Deleted old {collection_name}")
    except:
        print(f"  {collection_name} didn't exist")
    
    # Now create
    client.create_collection(
        collection_name=collection_name,
        vectors_config=models.VectorParams(
            size=384,
            distance=models.Distance.COSINE
        ),
        quantization_config=quantization_config
    )
    
    # Upload the same recipe data
    points = []
    for i, recipe in enumerate(my_domain_collection):
        vector = encoder.encode(recipe["description"]).tolist()
        points.append(models.PointStruct(id=i, vector=vector, payload=recipe))
    
    client.upload_points(collection_name=collection_name, points=points)
    
    print(f"✓ Created and populated {collection_name}")


✓ Deleted old quantized_scalar
✓ Created and populated quantized_scalar
✓ Deleted old quantized_binary
✓ Created and populated quantized_binary
✓ Deleted old quantized_product
✓ Created and populated quantized_product


## Step 3: Upload Data and Measure Impact

In [33]:
def benchmark(collection_name, your_test_queries, method_name):
    """Measure quantized search performance"""
    
    # Test without oversampling/rescoring first
    no_rescoring_metrics = measure_search_performance(
        collection_name, 
        your_test_queries, 
        f"{method_name} (No Rescoring)"
    )
    
    # Test with oversampling and rescoring
    def search_with_rescoring(collection_name, query_text, oversampling_factor=3.0):
        # Encode text to vector first
        query_vector = encoder.encode(query_text).tolist()
        
        start_time = time.time()
        
        response = client.query_points(
            collection_name=collection_name,
            query=query_vector,  # Use the encoded vector
            limit=10,
            search_params=models.SearchParams(
                quantization=models.QuantizationSearchParams(
                    rescore=True,
                    oversampling=oversampling_factor,
                )
            ),
        )
        
        return (time.time() - start_time) * 1000, response
    
    # Measure with rescoring
    rescoring_latencies = []
    for query_text in your_test_queries:
        latency, response = search_with_rescoring(collection_name, query_text)
        rescoring_latencies.append(latency)
    
    avg_rescoring = np.mean(rescoring_latencies)
    p95_rescoring = np.percentile(rescoring_latencies, 95)
    
    print(f"{method_name} (With Rescoring):")
    print(f"  Average latency: {avg_rescoring:.2f}ms")
    print(f"  P95 latency: {p95_rescoring:.2f}ms")
    
    return {
        "no_rescoring": no_rescoring_metrics,
        "with_rescoring": {"avg": avg_rescoring, "p95": p95_rescoring}
    }


In [36]:
# Test each quantization method
quantization_results = {}
for method_name in quantization_configs.keys():
    collection_name = f"quantized_{method_name}"
    print(f"\nBenchmarking {method_name}...")
    quantization_results[method_name] = benchmark(
        collection_name, my_test_queries, method_name
    )



Benchmarking scalar...
scalar (No Rescoring):
  Average latency: 206.41ms
  P95 latency: 194.37ms
scalar (With Rescoring):
  Average latency: 189.72ms
  P95 latency: 197.79ms

Benchmarking binary...
binary (No Rescoring):
  Average latency: 188.39ms
  P95 latency: 197.21ms
binary (With Rescoring):
  Average latency: 190.24ms
  P95 latency: 199.10ms

Benchmarking product...
product (No Rescoring):
  Average latency: 188.62ms
  P95 latency: 195.94ms
product (With Rescoring):
  Average latency: 190.51ms
  P95 latency: 199.68ms


## Step 4: Optimize Oversampling Factors

In [34]:
def measure_accuracy_retention(original_collection, quantized_collection, test_queries, factors=[2, 3, 5, 8, 10]):
    """Compare search results between original and quantized collections"""
    results = {}

    for factor in factors:
        accuracy_scores = []
        
        for query_text in test_queries:
            # Encode query to vector
            query_vector = encoder.encode(query_text).tolist()
            
            # Get baseline results
            baseline_results = client.query_points(
                collection_name=original_collection,
                query=query_vector,
                limit=10
            )
            baseline_ids = [point.id for point in baseline_results.points]

            # Get quantized results with rescoring
            quantized_results = client.query_points(
                collection_name=quantized_collection,
                query=query_vector,
                limit=10,
                search_params=models.SearchParams(
                    quantization=models.QuantizationSearchParams(
                        rescore=True,
                        oversampling=factor,
                    )
                ),
            )
            quantized_ids = [point.id for point in quantized_results.points]
            
            # Calculate overlap (simple accuracy measure)
            overlap = len(set(baseline_ids) & set(quantized_ids))
            accuracy = overlap / len(baseline_ids)
            accuracy_scores.append(accuracy)
        
        results[factor] = {
            "avg_accuracy": np.mean(accuracy_scores)
        }
    
    return results


def tune_oversampling(collection_name, test_queries, factors=[2, 3, 5, 8, 10]):
    """Find optimal oversampling factor"""
    results = {}
    
    for factor in factors:
        latencies = []
        
        for query_text in test_queries:
            # Encode query to vector
            query_vector = encoder.encode(query_text).tolist()
            
            start_time = time.time()
            
            response = client.query_points(
                collection_name=collection_name,
                query=query_vector,
                limit=10,
                search_params=models.SearchParams(
                    quantization=models.QuantizationSearchParams(
                        rescore=True,
                        oversampling=factor,
                    )
                ),
            )
            
            latencies.append((time.time() - start_time) * 1000)
        
        results[factor] = {
            "avg_latency": np.mean(latencies),
            "p95_latency": np.percentile(latencies, 95)
        }
    
    return results

# Tune oversampling for your method of choice
best_method = "binary"
oversampling_factors = [2, 3, 5, 8, 10]

oversampling_results_latency = tune_oversampling(
    f"quantized_{best_method}", 
    my_test_queries,  # ← Changed from your_test_queries
    oversampling_factors
)

oversampling_results_accuracy = measure_accuracy_retention(
    "my_domain_collection",  # ← Changed from your_domain_collection
    f"quantized_{best_method}", 
    my_test_queries,  # ← Changed from your_test_queries
    oversampling_factors
)

print("Oversampling Factor Optimization:")
for factor in oversampling_factors:
    print(f"  {factor}x:")
    print(f"    {oversampling_results_latency[factor]['avg_latency']:.2f}ms avg latency, {oversampling_results_latency[factor]['p95_latency']:.2f}ms P95 latency")
    print(f"    {oversampling_results_accuracy[factor]['avg_accuracy']:.2%} accuracy retention")


Oversampling Factor Optimization:
  2x:
    664.02ms avg latency, 1166.43ms P95 latency
    100.00% accuracy retention
  3x:
    683.99ms avg latency, 1007.04ms P95 latency
    100.00% accuracy retention
  5x:
    505.58ms avg latency, 997.15ms P95 latency
    100.00% accuracy retention
  8x:
    447.84ms avg latency, 788.28ms P95 latency
    100.00% accuracy retention
  10x:
    423.05ms avg latency, 753.09ms P95 latency
    100.00% accuracy retention


## Step 5: Analyze Your Results

In [37]:
print("=" * 60)
print("QUANTIZATION PERFORMANCE ANALYSIS")
print("=" * 60)

print(f"\nBaseline Performance:")
print(f"  Average latency: {baseline_metrics['avg']:.2f}ms")
print(f"  P95 latency: {baseline_metrics['p95']:.2f}ms")

print(f"\nQuantization Results:")
for method, results in quantization_results.items():
    no_rescoring = results['no_rescoring']
    with_rescoring = results['with_rescoring']
    
    speedup_no_rescoring = baseline_metrics['avg'] / no_rescoring['avg']
    speedup_with_rescoring = baseline_metrics['avg'] / with_rescoring['avg']
    
    print(f"\n{method.upper()}:")
    print(f"  Without rescoring: {no_rescoring['avg']:.2f}ms ({speedup_no_rescoring:.1f}x speedup)")
    print(f"  With rescoring: {with_rescoring['avg']:.2f}ms ({speedup_with_rescoring:.1f}x speedup)")

QUANTIZATION PERFORMANCE ANALYSIS

Baseline Performance:
  Average latency: 929.36ms
  P95 latency: 1975.89ms

Quantization Results:

SCALAR:
  Without rescoring: 206.41ms (4.5x speedup)
  With rescoring: 189.72ms (4.9x speedup)

BINARY:
  Without rescoring: 188.39ms (4.9x speedup)
  With rescoring: 190.24ms (4.9x speedup)

PRODUCT:
  Without rescoring: 188.62ms (4.9x speedup)
  With rescoring: 190.51ms (4.9x speedup)
